# LINC-seq TCR–pMHC NGS Analysis Pipeline

This notebook consolidates the analysis pipeline used to process paired-end
NGS reads from the combinatorial TCR–peptide library screen into enrichment-scored MHC–TCR pair count tables.

**Pipeline stages** (each is one section below, run in order):

1. Configuration
2. Read trimming & QC (adapter trimming, quality filtering, length trimming)
3. Read merging & anchor-sequence filtering
4. Translation & per-pair enrichment scoring
5. Count-table reformatting
6. Merging multiple samples into one table (with optional per-sample filtering)
7. (Optional) Enrich2 setup — citable, peer-reviewed enrichment scoring



## 1. Configuration

In [ ]:
import os
import subprocess
import itertools
from multiprocessing import Pool
from itertools import islice

import pandas as pd
import numpy as np
from scipy.stats import fisher_exact

# --------------------------------------------------------------------------
# Edit only this block to point the pipeline at a new sample.
# --------------------------------------------------------------------------
sample_name   = "SW_4"          # base name shared by <sample>_R1.fastq.gz / _R2.fastq.gz
data_dir      = "./data"        # directory containing the raw fastq.gz files
out_dir       = "./output"      # directory outputs are written to
num_cores     = 8

# Adapter sequences flanking the CDR3b (R1) and peptide (R2) reads
r1_adapter    = "AGCGCCTTCACAAAC"
r2_adapter    = "GCCTCCTCCTGAACC"

# Fixed-length windows kept after trimming (see 20260511 processing log)
r1_keep_len   = 48   # CDR3b read
r2_keep_len   = 27   # peptide read
trim_5p_r1    = 13   # bases trimmed from the 5' end of R1 after QC
trim_5p_r2    = 9    # bases trimmed from the 5' end of R2 after QC

# Conserved anchor sequences used to validate merged reads before scoring
# (positions are 0-indexed into the merged R1+revcomp(R2) sequence)
anchor_checks = {
    (0, 18):  "TGCGCCAGCAGGCCGGGC",
    (30, 48): "CAACCTGAACAATATTTT",
    (48, 66): "CTTTTGTTCGGGTATCCG",
    (72, 75): "GTT",
}

# Codon positions (within the merged/filtered read) translated into the
# 6-letter peptide+TCR identifier: pos0-1 = peptide, pos2 = gap, pos3-6 = TCR
codon_windows = [(18, 21), (21, 24), (24, 27), (27, 30), (66, 69), (69, 72)]

os.makedirs(out_dir, exist_ok=True)


## 2. Read trimming & QC

Adapter-trims paired reads (both mates in one Cutadapt call, so pairing is
never broken), quality-filters with fastp, then trims fixed-length windows
and reverse-complements the peptide read. Requires `cutadapt`, `fastp`, and
`seqtk` on PATH.

In [ ]:
def trim_and_qc(sample, data_dir=data_dir, out_dir=out_dir,
                 r1_adapter=r1_adapter, r2_adapter=r2_adapter,
                 r1_keep_len=r1_keep_len, r2_keep_len=r2_keep_len,
                 trim_5p_r1=trim_5p_r1, trim_5p_r2=trim_5p_r2,
                 cores=num_cores):
    """Adapter-trim, QC-filter, length-trim and reverse-complement one
    paired-end sample. Returns paths to the two ready-for-merge fastq.gz files."""
    r1_in = os.path.join(data_dir, f"{sample}_R1.fastq.gz")
    r2_in = os.path.join(data_dir, f"{sample}_R2.fastq.gz")

    r1_trim = os.path.join(out_dir, f"{sample}_trim_R1.fastq.gz")
    r2_trim = os.path.join(out_dir, f"{sample}_trim_R2.fastq.gz")
    subprocess.run([
        "cutadapt", "-j", str(cores),
        "-g", r1_adapter, "-G", r2_adapter, "--discard-untrimmed",
        "-o", r1_trim, "-p", r2_trim, r1_in, r2_in,
    ], check=True)

    r1_qc = os.path.join(out_dir, f"{sample}_trim_R1_qc.fastq.gz")
    r2_qc = os.path.join(out_dir, f"{sample}_trim_R2_qc.fastq.gz")
    subprocess.run([
        "fastp", "-i", r1_trim, "-I", r2_trim, "-o", r1_qc, "-O", r2_qc,
        "-q", "20", "-u", "20", "--thread", str(cores),
    ], check=True)

    r1_trim2 = os.path.join(out_dir, f"{sample}_trim2_R1.fastq.gz")
    r2_trim2 = os.path.join(out_dir, f"{sample}_trim2_R2.fastq.gz")
    subprocess.run([
        "cutadapt", "-j", str(cores), "-u", str(trim_5p_r1), "-U", str(trim_5p_r2),
        "-o", r1_trim2, "-p", r2_trim2, r1_qc, r2_qc,
    ], check=True)

    r1_final = os.path.join(out_dir, f"{sample}_R1_CDR.fastq.gz")
    r2_final = os.path.join(out_dir, f"{sample}_R2_pep.fastq.gz")
    subprocess.run(["cutadapt", "-j", str(cores), "-l", str(r1_keep_len),
                     "-o", r1_final, r1_trim2], check=True)
    subprocess.run(["cutadapt", "-j", str(cores), "-l", str(r2_keep_len),
                     "-o", r2_final, r2_trim2], check=True)

    return r1_final, r2_final

# Example: trim_and_qc(sample_name)


## 3. Read merging & anchor-sequence filtering

Pairs R1 (CDR3b) with the reverse complement of R2 (peptide) and keeps only
reads that match the conserved vector-backbone anchor sequences, confirming
correct read orientation and library structure before any scoring.

In [ ]:
def reverse_complement(seq):
    table = str.maketrans("ATCGNatcgn", "TAGCNtagcn")
    return seq.translate(table)[::-1]

def passes_anchor_filter(seq, anchor_checks=anchor_checks):
    return all(seq[start:end] == expected for (start, end), expected in anchor_checks.items())

def _read_fastq_pigz(filename):
    proc = subprocess.Popen(["pigz", "-dc", filename], stdout=subprocess.PIPE, text=True)
    while True:
        lines = [proc.stdout.readline().strip() for _ in range(4)]
        if not lines[0]:
            break
        yield lines
    proc.stdout.close()
    proc.wait()

def _chunks(iterable, size):
    it = iter(iterable)
    while True:
        chunk = list(islice(it, size))
        if not chunk:
            break
        yield chunk

def _merge_chunk(chunk):
    out = []
    for r1, r2 in chunk:
        seq = r1[1] + reverse_complement(r2[1])
        if passes_anchor_filter(seq):
            qual = r1[3] + r2[3][::-1]
            out.append(f"{r1[0].split()[0]}\n{seq}\n+\n{qual}\n")
    return "".join(out)

def merge_and_filter_reads(sample, r1_path, r2_path, out_dir=out_dir,
                            cores=num_cores, chunk_size=50000):
    """Merge R1/revcomp(R2) and drop reads that fail the anchor filter.
    Returns the path to the merged, filtered fastq.gz."""
    out_path = os.path.join(out_dir, f"{sample}_merged_filtered.fastq.gz")
    r1_reads = _read_fastq_pigz(r1_path)
    r2_reads = _read_fastq_pigz(r2_path)
    paired = zip(r1_reads, r2_reads)

    with open(out_path, "wb") as out_file:
        pigz_out = subprocess.Popen(["pigz", "-p", str(cores), "-c"],
                                     stdin=subprocess.PIPE, stdout=out_file, text=True)
        with Pool(processes=cores) as pool:
            for result in pool.imap(_merge_chunk, _chunks(paired, chunk_size), chunksize=2):
                pigz_out.stdin.write(result)
        pigz_out.stdin.close()
        pigz_out.wait()
    return out_path

# Example: merge_and_filter_reads(sample_name, r1_final, r2_final)


## 4. Translation & per-pair enrichment scoring

Translates the six codon windows into the peptide+TCR identifier, drops any
sequence containing a stop codon, and computes a one-sided Fisher's exact
test p-value for each identifier's enrichment (replacing the lab-internal
`goscripts.enrichment_stats` call with the equivalent `scipy.stats.fisher_exact`
so this step has no non-public dependencies).

In [ ]:
CODON_TABLE = {
    'TTT':'F','TTC':'F','TTA':'L','TTG':'L','CTT':'L','CTC':'L','CTA':'L','CTG':'L',
    'ATT':'I','ATC':'I','ATA':'I','ATG':'M','GTT':'V','GTC':'V','GTA':'V','GTG':'V',
    'TCT':'S','TCC':'S','TCA':'S','TCG':'S','CCT':'P','CCC':'P','CCA':'P','CCG':'P',
    'ACT':'T','ACC':'T','ACA':'T','ACG':'T','GCT':'A','GCC':'A','GCA':'A','GCG':'A',
    'TAT':'Y','TAC':'Y','TAA':'*','TAG':'*','CAT':'H','CAC':'H','CAA':'Q','CAG':'Q',
    'AAT':'N','AAC':'N','AAA':'K','AAG':'K','GAT':'D','GAC':'D','GAA':'E','GAG':'E',
    'TGT':'C','TGC':'C','TGA':'*','TGG':'W','CGT':'R','CGC':'R','CGA':'R','CGG':'R',
    'AGT':'S','AGC':'S','AGA':'R','AGG':'R','GGT':'G','GGC':'G','GGA':'G','GGG':'G',
}

def translate(seq):
    return "".join(CODON_TABLE.get(seq[i:i+3], "X") for i in range(0, len(seq) - 2, 3))

def translate_and_score(sample, merged_fastq_path, out_dir=out_dir,
                         codon_windows=codon_windows, p_cutoff=0.05, cores=num_cores):
    """Translate merged reads, drop stop-codon sequences, and score each
    identifier's enrichment with a one-sided Fisher's exact test.
    Writes '<sample>_enrichment_pvalues.txt' (Full Sequence, Count, P-Value)."""
    count_full, count_pep, count_tcr = {}, {}, {}
    reads = _read_fastq_pigz(merged_fastq_path)

    for _, sequence in reads:
        codons = [translate(sequence[s:e]) for s, e in codon_windows]
        full_seq = "".join(codons[4:6]) + "".join(codons[0:4])  # peptide + TCR
        if "*" in full_seq:
            continue
        pep_seq, tcr_seq = full_seq[:2], full_seq[2:]
        count_full[full_seq] = count_full.get(full_seq, 0) + 1
        count_pep[pep_seq]   = count_pep.get(pep_seq, 0) + 1
        count_tcr[tcr_seq]   = count_tcr.get(tcr_seq, 0) + 1

    total = sum(count_full.values())
    rows = []
    for full_seq, n_ab in count_full.items():
        pep_seq, tcr_seq = full_seq[:2], full_seq[2:]
        n_pep = count_pep[pep_seq]
        n_tcr = count_tcr[tcr_seq]
        # one-sided Fisher's exact test: is this pair enriched relative to
        # its marginal peptide/TCR frequencies?
        table = [[n_ab, n_pep - n_ab], [n_tcr - n_ab, total - n_pep - n_tcr + n_ab]]
        _, pval = fisher_exact(table, alternative="greater")
        rows.append((full_seq, n_ab, pval))

    out_path = os.path.join(out_dir, f"{sample}_enrichment_pvalues.txt")
    pd.DataFrame(rows, columns=["Full Sequence", "Count", "P-Value"]).to_csv(
        out_path, sep="\t", index=False)
    print(f"{sample}: {len(rows)} identifiers scored, "
          f"{sum(p < p_cutoff for *_, p in rows)} pass p < {p_cutoff}")
    return out_path

# Example: translate_and_score(sample_name, merged_path)


## 5. Count-table reformatting

Splits a two-column `sequence\tread_count` file into the standard three-column
`MHC\tTCR\tread_count` table used by every downstream step.

In [ ]:
def reformat_dataset(input_file, output_file, mhc_len=2):
    """Split a 'Sequence  read_count' table into 'MHC  TCR  read_count'."""
    data = pd.read_csv(input_file, sep="\t", header=None, names=["Sequence", "read_count"])
    data["MHC"] = data["Sequence"].str[:mhc_len]
    data["TCR"] = data["Sequence"].str[mhc_len:]
    reformatted = data[["MHC", "TCR", "read_count"]]
    reformatted.to_csv(output_file, sep="\t", index=False)
    print(f"Reformatted dataset saved to: {output_file}")
    return reformatted

# Example: reformat_dataset("sequence_counts.txt", "SW_4_reformat.txt")


## 6. Merging multiple samples into one table

Joins any number of `MHC  TCR  read_count` tables on the (MHC, TCR) pair,
one read-count column per input file, missing pairs filled with 0.
Per-sample minimum-count filtering is optional (`filters=None` skips it).

In [ ]:
def merge_mhc_tcr_datasets(input_files, output_file, filters=None):
    """Merge MHC-TCR count tables on (MHC, TCR); optionally apply a
    per-file minimum read-count threshold before saving.

    filters: dict mapping input filename -> minimum count (or None to skip
             that file / skip filtering entirely).
    """
    data_by_pair = {}
    col_names = []

    for i, file in enumerate(input_files, start=1):
        print(f"Reading file: {file}")
        try:
            df = pd.read_csv(file, sep=None, engine="python")
        except Exception:
            df = pd.read_csv(file, sep="\t")
        df.columns = [c.strip() for c in df.columns]

        expected = {"MHC", "TCR", "read_count"}
        if not expected.issubset(set(df.columns)):
            if df.shape[1] == 3:
                df.columns = ["MHC", "TCR", "read_count"]
            else:
                print(f"  Skipping {file}: incompatible format {df.columns.tolist()}")
                continue
        df = df[["MHC", "TCR", "read_count"]]

        col_name = f"read_count_from_{os.path.basename(file)}"
        col_names.append(col_name)
        for _, row in df.iterrows():
            key = (row["MHC"], row["TCR"])
            entry = data_by_pair.setdefault(key, {"MHC": row["MHC"], "TCR": row["TCR"]})
            entry[col_name] = row["read_count"]

    result_df = pd.DataFrame(list(data_by_pair.values())).fillna(0)
    for col in col_names:
        if col in result_df.columns:
            result_df[col] = result_df[col].astype(int)

    if filters:
        for col, threshold in filters.items():
            if threshold is not None and col in result_df.columns:
                result_df = result_df[result_df[col] >= threshold]

    result_df = result_df.sort_values(["MHC", "TCR"])
    result_df.to_csv(output_file, index=False, sep="\t")
    print(f"Merged data saved to {output_file} ({len(result_df)} MHC-TCR pairs)")
    return result_df

# Example (no filtering):
#   merge_mhc_tcr_datasets(["SW_2_reformat.txt", "SW_3_reformat.txt"], "merge_SW23_file.txt")
# Example (with a minimum-count filter on one sample):
#   merge_mhc_tcr_datasets(
#       ["SW_2_reformat.txt", "SW_3_reformat.txt"], "merge_SW23_filtered.txt",
#       filters={"read_count_from_SW_2_reformat.txt": 5},
#   )


## 7. Enrich2 setup

Prepares the merged initial-vs-selected count tables for
[Enrich2](https://github.com/FowlerLab/Enrich2) (Rubin et al. 2017, *Genome
Biology*) — a peer-reviewed, citable scoring method, as an alternative /
cross-check to the Fisher's-exact scoring in Section 4. This step only
writes the Enrich2 input files and config; run Enrich2 itself separately
(`pip install enrich2`, then `enrich_cmd config.json WLS full --output-dir ...`).

In [ ]:
import json

def _load_pair_counts(path):
    df = pd.read_csv(path, sep="\t")
    identifier = df["MHC"].astype(str) + "_" + df["TCR"].astype(str)
    counts = pd.Series(df["read_count"].values, index=identifier, name="count")
    if counts.index.duplicated().any():
        raise ValueError(f"Duplicate MHC_TCR identifiers in {path}; aggregate first.")
    return counts

def setup_enrich2_analysis(initial_file, final_file, run_dir, name=None):
    """Write Enrich2-format counts files + config.json for one initial-vs-final
    comparison. Returns the config path."""
    run_dir = os.path.abspath(run_dir)
    counts_dir = os.path.join(run_dir, "counts")
    os.makedirs(counts_dir, exist_ok=True)

    initial_counts = _load_pair_counts(initial_file)
    final_counts = _load_pair_counts(final_file)

    initial_out = os.path.join(counts_dir, "initial_counts.tsv")
    final_out = os.path.join(counts_dir, "final_counts.tsv")
    initial_counts.to_frame().to_csv(initial_out, sep="\t", index=True, index_label="")
    final_counts.to_frame().to_csv(final_out, sep="\t", index=True, index_label="")

    if name is None:
        name = (f"{os.path.splitext(os.path.basename(initial_file))[0]}_vs_"
                f"{os.path.splitext(os.path.basename(final_file))[0]}")

    config = {
        "name": name,
        "output directory": run_dir,
        "libraries": [
            {"name": "initial", "timepoint": 0, "identifiers": {}, "counts file": initial_out},
            {"name": "final", "timepoint": 1, "identifiers": {}, "counts file": final_out},
        ],
    }
    config_path = os.path.join(run_dir, "config.json")
    with open(config_path, "w") as f:
        json.dump(config, f, indent=2)

    print(f"Analysis name: {name}")
    print(f"Wrote Enrich2 config to: {config_path}")
    print(f"Run with: enrich_cmd {config_path} WLS full --output-dir {run_dir}")
    return config_path

# Example: setup_enrich2_analysis("SW_2_reformat.txt", "SW_4_reformat.txt", "enrich2_run")


## End-to-end example

Runs the full pipeline on `sample_name` from Section 1 (requires raw
`<sample>_R1.fastq.gz` / `<sample>_R2.fastq.gz` files in `data_dir`).

In [ ]:
# r1_final, r2_final = trim_and_qc(sample_name)
# merged_path = merge_and_filter_reads(sample_name, r1_final, r2_final)
# pvalue_path = translate_and_score(sample_name, merged_path)
# reformat_dataset(pvalue_path, os.path.join(out_dir, f"{sample_name}_reformat.txt"))
